# TourismGPT — Chat UI Demo
Gradio interface for the fine-tuned Phi-3 Mini + Wikivoyage RAG pipeline.

**Prerequisites:** Run `rag_pipeline.ipynb` first to build the FAISS index and save the adapter to Drive.

In [1]:
# CELL 1 — Install dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install sentence-transformers faiss-gpu gradio

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-2iuaoffg/unsloth_0ef02d65bc374ac9ab4a337825507a13
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-2iuaoffg/unsloth_0ef02d65bc374ac9ab4a337825507a13
  Resolved https://github.com/unslothai/unsloth.git to commit 1ef3f64d491d7e483592bd0e5b548c2e14d07d52
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 139.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 119.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 20.8 MB/s eta 0:00:00
  

In [2]:
# CELL 2 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# CELL 3 — Imports and config
import bz2, re, os, pickle
import numpy as np
import torch
from xml.etree import ElementTree as ET

ADAPTER_DIR  = "/content/drive/MyDrive/tourism_gpt_adapter"
DRIVE_INDEX  = "/content/drive/MyDrive/wikivoyage_index"
EMBED_MODEL  = "sentence-transformers/all-MiniLM-L6-v2"
TOP_K        = 3
MAX_SEQ_LEN  = 2048
MAX_NEW_TOKENS = 400

print("Config ready ✓")

Config ready ✓


In [4]:
# CELL 4 — Load FAISS index + chunks from Drive
import faiss
from sentence_transformers import SentenceTransformer

print("Loading embedding model ...")
embedder = SentenceTransformer(EMBED_MODEL)

print("Loading FAISS index ...")
index = faiss.read_index(f"{DRIVE_INDEX}/wikivoyage.index")
with open(f"{DRIVE_INDEX}/chunks.pkl", "rb") as f:
    all_chunks = pickle.load(f)

print(f"Index: {index.ntotal} vectors  |  Chunks: {len(all_chunks)}")

Loading embedding model ...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loading FAISS index ...
Index: 18785 vectors  |  Chunks: 18785


In [5]:
# CELL 5 — Load fine-tuned Phi-3 + LoRA adapter
from unsloth import FastLanguageModel

print("Loading Phi-3 Mini + LoRA adapter ...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = ADAPTER_DIR,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = True,
)
FastLanguageModel.for_inference(model)
print("Model ready ✓")

/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:153: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading Phi-3 Mini + LoRA adapter ...
==((====))==  Unsloth 2026.6.7: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/194 [00:00<?, ?B/s]

...

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/458 [00:00<?, ?B/s]

Unsloth 2026.6.7 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Model ready ✓


In [6]:
# CELL 6 — RAG functions
def retrieve(query, k=TOP_K):
    q_emb = embedder.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q_emb)
    scores, indices = index.search(q_emb, k)
    return [{**all_chunks[idx], "score": float(score)}
            for score, idx in zip(scores[0], indices[0])]

def ask_rag(question):
    chunks = retrieve(question)
    context = "\n\n".join(
        f"[{i}] {c['title']}: {c['text']}" for i, c in enumerate(chunks, 1)
    )
    prompt = (
        f"<|user|>\n"
        f"You are TourismGPT, an expert travel assistant. "
        f"Use the context below to give a helpful, specific answer. "
        f"If the context does not cover the question, answer from your training knowledge.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}<|end|>\n"
        f"<|assistant|>\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    max_input = MAX_SEQ_LEN - MAX_NEW_TOKENS
    if inputs["input_ids"].shape[1] > max_input:
        inputs = {k: v[:, -max_input:] for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens = MAX_NEW_TOKENS,
            temperature    = 0.7,
            top_p          = 0.9,
            do_sample      = True,
            pad_token_id   = tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    sources = list({c["title"] for c in chunks})
    return answer, sources

print("RAG functions ready ✓")

RAG functions ready ✓


In [7]:
# CELL 7 — Launch Gradio chat UI
import gradio as gr

EXAMPLE_QUESTIONS = [
    "Plan a 5-day itinerary for Tokyo on a budget.",
    "Compare Bali vs Thailand for a honeymoon.",
    "What is the estimated budget for a week in Paris?",
    "What cultural customs should I know before visiting Japan?",
    "What are the best things to do in Rome?",
    "Is Bali safe for solo female travellers?",
]

def chat(message, history):
    if not message.strip():
        return history, "", ""
    answer, sources = ask_rag(message)
    history = history or []
    history.append((message, answer))
    source_text = "**Sources:** " + " · ".join(f"`{s}`" for s in sources)
    return history, "", source_text

with gr.Blocks(title="TourismGPT", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "# ✈️ TourismGPT\n"
        "**Fine-tuned Phi-3 Mini + Wikivoyage RAG** — Ask anything about travel!"
    )

    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(height=460, label="TourismGPT")
            with gr.Row():
                msg = gr.Textbox(
                    placeholder="Ask a travel question ...",
                    show_label=False,
                    scale=5,
                )
                send_btn = gr.Button("Send", variant="primary", scale=1)
            sources_box = gr.Markdown("", label="Sources")
            gr.ClearButton([chatbot, msg, sources_box], value="Clear chat")

        with gr.Column(scale=1):
            gr.Markdown("### Try these questions")
            for q in EXAMPLE_QUESTIONS:
                gr.Button(q, size="sm").click(
                    fn=lambda x=q: x, outputs=msg
                )

    send_btn.click(chat, [msg, chatbot], [chatbot, msg, sources_box])
    msg.submit(chat, [msg, chatbot], [chatbot, msg, sources_box])

demo.launch(share=True, debug=False)

/tmp/ipykernel_2572/2052684328.py:22: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="TourismGPT", theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_2572/2052684328.py:30: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=460, label="TourismGPT")
/tmp/ipykernel_2572/2052684328.py:30: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=460, label="TourismGPT")


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d1bf32c52b39335bc6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
